In [ ]:
# [0] Colab setup — clone repo so src/ is available, install CLIP
import sys, os

REPO = 'https://github.com/sudikshyapant/Sparse-CLIP-with-Spectral-Loss'
REPO_DIR = '/content/Sparse-CLIP-with-Spectral-Loss'

if 'google.colab' in sys.modules:
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {REPO} {REPO_DIR}')
    os.chdir(REPO_DIR)
    os.system('pip install -q git+https://github.com/openai/CLIP.git')

repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print('Working dir:', repo_root)

# Variation 2 — Kernel InfoNCE (Gaussian / Polynomial / Mixture)

In [ ]:
# [1] Config — mounts Drive, sets all paths
from google.colab import drive
drive.mount('/content/drive')

from src.config import CONFIG
print('cache_dir :', CONFIG['cache_dir'])
print('coco_root :', CONFIG['coco_root'])
print('device    :', CONFIG['device'])

In [ ]:
# [2] Download COCO images to local storage (skipped if cache already on Drive)
#
# First run:  downloads images locally → computes embeddings → saves .pt to Drive
# Later runs: cache found on Drive → this cell is a no-op

import os, pathlib

DRIVE_COCO = '/content/drive/MyDrive/sparse_clip/coco'
LOCAL_COCO = '/content/coco'
cache_dir  = CONFIG['cache_dir']

need_train = not (cache_dir / 'train_img_emb.pt').exists()
need_val   = not (cache_dir / 'val_img_emb.pt').exists()

if not need_train and not need_val:
    print('Cache found on Drive — skipping all downloads.')
else:
    os.makedirs(f'{LOCAL_COCO}/annotations', exist_ok=True)

    # Annotations — try Drive first, fall back to direct download
    ann_needed = []
    for f in ['captions_train2017.json', 'captions_val2017.json']:
        dst = f'{LOCAL_COCO}/annotations/{f}'
        if not os.path.exists(dst):
            drive_src = f'{DRIVE_COCO}/annotations/{f}'
            if os.path.exists(drive_src):
                os.system(f'cp {drive_src} {dst}')
                print(f'Copied {f}')
            else:
                ann_needed.append(f)
    if ann_needed:
        print('Downloading annotations (~240 MB)...')
        os.system('wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O /tmp/ann.zip')
        os.system(f'unzip -jo /tmp/ann.zip "annotations/captions_*.json" -d {LOCAL_COCO}/annotations')
        os.system('rm /tmp/ann.zip')
        print('Annotations ready')

    if need_train:
        print('Downloading train2017 (~18 GB) to local storage...')
        os.system('wget -q http://images.cocodataset.org/zips/train2017.zip -O /tmp/train2017.zip')
        os.system(f'unzip -q /tmp/train2017.zip -d {LOCAL_COCO}')
        os.system('rm /tmp/train2017.zip')
        print('train2017 ready')

    if need_val:
        if os.path.exists(f'{DRIVE_COCO}/val2017'):
            print('Copying val2017 from Drive...')
            os.system(f'cp -r {DRIVE_COCO}/val2017 {LOCAL_COCO}/val2017')
        else:
            print('Downloading val2017 (~1 GB)...')
            os.system('wget -q http://images.cocodataset.org/zips/val2017.zip -O /tmp/val2017.zip')
            os.system(f'unzip -q /tmp/val2017.zip -d {LOCAL_COCO}')
            os.system('rm /tmp/val2017.zip')
        print('val2017 ready')

    # Point CONFIG to local images for fast reading
    CONFIG['coco_train_images'] = pathlib.Path(LOCAL_COCO) / 'train2017'
    CONFIG['coco_val_images']   = pathlib.Path(LOCAL_COCO) / 'val2017'
    CONFIG['coco_train_ann']    = pathlib.Path(LOCAL_COCO) / 'annotations' / 'captions_train2017.json'
    CONFIG['coco_val_ann']      = pathlib.Path(LOCAL_COCO) / 'annotations' / 'captions_val2017.json'
    print('CONFIG paths updated to local storage')

In [ ]:
# [3] Compute and cache CLIP embeddings
# Reads images locally (first run) or loads .pt from Drive (subsequent runs).
import clip, torch
from src.data_utils import cache_or_compute_embeddings, make_loader

device = CONFIG['device']
clip_model, preprocess = clip.load(CONFIG['clip_model'], device=device)
clip_model.eval()

train_img, train_txt = cache_or_compute_embeddings(clip_model, preprocess, 'train', CONFIG)
val_img,   val_txt   = cache_or_compute_embeddings(clip_model, preprocess, 'val',   CONFIG)
print(f'train: {train_img.shape}  val: {val_img.shape}')

train_loader = make_loader(train_img, train_txt, CONFIG['batch_size'])

In [ ]:
# [4] Model factory
from src.model import SparseHead

def make_head():
    return SparseHead(CONFIG['embed_dim'], CONFIG['sparse_dim']).to(device)

In [ ]:
# [4] Train one model per kernel
import torch.optim as optim
from functools import partial
from src.losses import gaussian_kernel, poly_kernel, MixtureKernel, kernel_infonce_loss
from src.train  import train_one_epoch, evaluate, save_checkpoint, make_run_tag

run_tag = make_run_tag(CONFIG['epochs'], CONFIG['batch_size'])
print(f'Run tag: {run_tag}')

mixture_kernel = MixtureKernel(alpha_init=0.5).to(device)

kernels = {
    'gaussian': (gaussian_kernel,  []),
    'poly':     (poly_kernel,      []),
    'mixture':  (mixture_kernel,   list(mixture_kernel.parameters())),
}

all_metrics   = {}
trained_heads = {}   # keep trained models for evaluation cells below

for kernel_name, (kfn, extra_params) in kernels.items():
    print(f'\n─── Kernel: {kernel_name} ───')
    head = make_head()
    loss_fn = partial(kernel_infonce_loss, kernel_fn=kfn)

    opt = optim.AdamW(
        list(head.parameters()) + extra_params,
        lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay']
    )

    loss_curve = []
    for epoch in range(1, CONFIG['epochs'] + 1):
        l = train_one_epoch(head, train_loader, opt, loss_fn, device)
        loss_curve.append(l)
        extra = f'  α={mixture_kernel.alpha.item():.3f}' if kernel_name == 'mixture' else ''
        print(f'  epoch {epoch:3d}/{CONFIG["epochs"]}  loss={l:.4f}{extra}')

    metrics = evaluate(head, val_img, val_txt, CONFIG)
    metrics['loss_curve'] = loss_curve
    if kernel_name == 'mixture':
        metrics['final_alpha'] = mixture_kernel.alpha.item()
    all_metrics[kernel_name]   = metrics
    trained_heads[kernel_name] = head
    print(metrics)
    save_checkpoint(head, metrics, kernel_name, 'variation2', CONFIG, run_tag)

In [ ]:
# [4b] Load variation1 baselines (InfoNCE + Spectral) matching run_tag
import json

baselines = {}
k = CONFIG['retrieval_k']
for bname in ['infonce', 'spectral']:
    path = CONFIG['results_dir'] / 'variation1' / f'{bname}_{run_tag}_metrics.json'
    if path.exists():
        with open(path) as f:
            baselines[bname] = json.load(f)
        m = baselines[bname]
        print(f'Loaded v1 {bname} ({run_tag}):  '
              f'IR@{k}={m[f"IR@{k}"]:.3f}  TR@{k}={m[f"TR@{k}"]:.3f}  '
              f'clarity={m["clarity"]:.4f}  cross_modal={m["cross_modal"]:.4f}')
    else:
        print(f'No v1 {bname} baseline at {path}  (run variation1 with {run_tag} to enable)')

if not baselines:
    print('No baselines loaded — plots will show kernel results only.')

In [ ]:
# [5] Results table
k = CONFIG['retrieval_k']
print(f'{"Model":14s}  IR@1   TR@1   IR@5   TR@5   L0_img  Active%  Clarity  Cross-modal')
for name, m in all_metrics.items():
    print(f"{name:14s}  {m[f'IR@{k}']:.3f}   {m[f'TR@{k}']:.3f}   "
          f"{m.get('IR@5', float('nan')):.3f}   {m.get('TR@5', float('nan')):.3f}   "
          f"{m['l0_img']:6.1f}  {m.get('active_pct', float('nan')):5.1f}%  "
          f"{m['clarity']:.4f}   {m['cross_modal']:.4f}")
for bname, m in baselines.items():
    print(f"{'v1_' + bname:14s}  {m[f'IR@{k}']:.3f}   {m[f'TR@{k}']:.3f}   "
          f"{m.get('IR@5', float('nan')):.3f}   {m.get('TR@5', float('nan')):.3f}   "
          f"{m['l0_img']:6.1f}  {m.get('active_pct', float('nan')):5.1f}%  "
          f"{m['clarity']:.4f}   {m['cross_modal']:.4f}  ← v1 baseline")
if 'final_alpha' in all_metrics.get('mixture', {}):
    print(f"\nMixture final α = {all_metrics['mixture']['final_alpha']:.4f}")

In [ ]:
# [6] Comparison plot
import matplotlib.pyplot as plt

names = list(all_metrics.keys())
k = CONFIG['retrieval_k']

# Per-kernel line styles for loss curves
KERNEL_STYLE = {'gaussian': '-', 'poly': '--', 'mixture': '-.'}
# Per-baseline reference style: infonce=dashed, spectral=dotted
BASELINE_STYLE = {'infonce': ('--', 'InfoNCE (v1)'), 'spectral': (':', 'Spectral (v1)')}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

# — Loss curves: different line type per kernel
for name, m in all_metrics.items():
    axes[0].plot(m['loss_curve'], label=name, linestyle=KERNEL_STYLE.get(name, '-'))
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

# — Retrieval@K bars + baseline reference lines
x = list(range(len(names)))
ir = [all_metrics[n][f'IR@{k}'] for n in names]
tr = [all_metrics[n][f'TR@{k}'] for n in names]
axes[1].bar([i - 0.2 for i in x], ir, 0.4, label=f'IR@{k}')
axes[1].bar([i + 0.2 for i in x], tr, 0.4, label=f'TR@{k}')
for bname, m in baselines.items():
    ls, label = BASELINE_STYLE[bname]
    axes[1].axhline(m[f'IR@{k}'], color='steelblue', linestyle=ls, linewidth=1.2,
                    label=f'{label} IR@{k}')
    axes[1].axhline(m[f'TR@{k}'], color='orange', linestyle=ls, linewidth=1.2,
                    label=f'{label} TR@{k}')
axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=10)
axes[1].set_title(f'Retrieval@{k}'); axes[1].legend(fontsize=8)

# — Clarity vs L0 (twin y-axis) + baseline reference lines
cl = [all_metrics[n]['clarity'] for n in names]
l0 = [all_metrics[n]['l0_img']  for n in names]
ax2 = axes[2].twinx()
axes[2].bar([i - 0.2 for i in x], cl, 0.4, color='steelblue', label='Clarity')
ax2.bar(    [i + 0.2 for i in x], l0, 0.4, color='orange',    label='L0 img')
for bname, m in baselines.items():
    ls, label = BASELINE_STYLE[bname]
    axes[2].axhline(m['clarity'], color='steelblue', linestyle=ls, linewidth=1.2, alpha=0.8)
    ax2.axhline(m['l0_img'],     color='orange',     linestyle=ls, linewidth=1.2, alpha=0.8)
axes[2].set_xticks(x); axes[2].set_xticklabels(names, rotation=10)
axes[2].set_ylabel('Clarity', color='steelblue')
ax2.set_ylabel('L0', color='orange')
bl_note = '  (dashed=InfoNCE, dotted=Spectral)' if baselines else ''
axes[2].set_title(f'Clarity vs L0{bl_note}')
axes[2].legend(loc='upper left'); ax2.legend(loc='upper right')

# — Cross-modal score
cm = [all_metrics[n]['cross_modal'] for n in names]
bars = axes[3].bar(x, cm, 0.5, color='mediumseagreen')
axes[3].axhline(0.5, color='red', linestyle='--', linewidth=1, label='ideal (0.5)')
for bname, m in baselines.items():
    ls, label = BASELINE_STYLE[bname]
    axes[3].axhline(m['cross_modal'], color='grey', linestyle=ls, linewidth=1.5, label=label)
axes[3].set_xticks(x); axes[3].set_xticklabels(names, rotation=10)
axes[3].set_ylim(0, 1)
axes[3].set_ylabel('Cross-modal score')
axes[3].set_title('Cross-modal balance')
axes[3].legend(fontsize=8)
for bar, v in zip(bars, cm):
    axes[3].text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.3f}',
                 ha='center', va='bottom', fontsize=9)

fig.tight_layout()
out_path = CONFIG['results_dir'] / 'variation2' / f'comparison_{run_tag}.png'
fig.savefig(out_path, dpi=150)
plt.show()
print(f'Saved {out_path}')

In [ ]:
# [7] CIFAR-100 zero-shot classification
import torch.nn.functional as F
from src.data_utils import load_or_compute_cifar100_embs
from src.metrics    import zero_shot_accuracy
from src.model      import SparseHead

cifar_img, cifar_cls, cifar_labels = load_or_compute_cifar100_embs(
    clip_model, preprocess, CONFIG['cache_dir'], device
)

# Baseline: raw CLIP (no SparseHead)
with torch.no_grad():
    raw_sims = cifar_img.to(device) @ cifar_cls.to(device).T
raw_acc = (raw_sims.argmax(1).cpu() == cifar_labels).float().mean().item()

print(f'\nCIFAR-100 Zero-Shot Accuracy (Acc@1)')
print(f'{"Model":16s}  Acc@1   vs raw CLIP')
print(f'{"CLIP (raw)":16s}  {raw_acc:.3f}   baseline')
for name, head_model in trained_heads.items():
    acc   = zero_shot_accuracy(head_model, cifar_cls, cifar_img, cifar_labels, device)
    delta = acc - raw_acc
    print(f'{name:16s}  {acc:.3f}   {"+" if delta >= 0 else ""}{delta:.3f}')

# Optionally load and evaluate variation1 baseline heads from saved checkpoints
for bname in baselines:
    ckpt = CONFIG['results_dir'] / 'variation1' / f'{bname}_{run_tag}_model.pt'
    if ckpt.exists():
        bl_head = SparseHead(CONFIG['embed_dim'], CONFIG['sparse_dim']).to(device)
        bl_head.load_state_dict(torch.load(ckpt, map_location=device))
        acc   = zero_shot_accuracy(bl_head, cifar_cls, cifar_img, cifar_labels, device)
        delta = acc - raw_acc
        print(f'{"v1_" + bname:16s}  {acc:.3f}   {"+" if delta >= 0 else ""}{delta:.3f}  ← v1 baseline')
    else:
        print(f'{"v1_" + bname:16s}  (checkpoint not found at {run_tag})')

In [ ]:
# [8] Modality score distribution for all kernels
import matplotlib.pyplot as plt
from src.visualization import plot_modality_grid

modality_data = {}
for name, head_model in trained_heads.items():
    head_model.eval()
    with torch.no_grad():
        z_img, _ = head_model(val_img.to(device))
        z_txt, _ = head_model(val_txt.to(device))
    modality_data[name] = (z_img.cpu(), z_txt.cpu())

fig = plot_modality_grid(modality_data, ncols=3,
                         out_path=CONFIG['results_dir'] / 'variation2' / f'modality_{run_tag}.png')
plt.show()
print('Saved modality distribution plot.')

In [ ]:
# [9] Feature activation heatmap + retrieval similarity matrix for all kernels
from src.visualization import plot_eval_grid

eval_data = {}
for name, head_model in trained_heads.items():
    head_model.eval()
    with torch.no_grad():
        z_img, img_z = head_model(val_img.to(device))
        z_txt, txt_z = head_model(val_txt.to(device))
    eval_data[name] = {
        'z_img': z_img.cpu(), 'z_txt': z_txt.cpu(),
        'img_z': img_z.cpu(), 'txt_z': txt_z.cpu(),
    }

fig = plot_eval_grid(eval_data,
                     out_path=CONFIG['results_dir'] / 'variation2' / f'eval_grid_{run_tag}.png')
plt.show()
print('Saved evaluation grid (modality / heatmap / similarity matrix).')